# Topic: SQL: Top N Records

## Definition (30-second explanation)
* The Top N Records pattern retrieves the highest (or lowest) N values either from an entire table or within specific groups.
* **Global Top N:** Uses `ORDER BY` paired with `LIMIT` to restrict the total rows returned.
* **Grouped Top N:** Uses window functions like `ROW_NUMBER()`, `RANK()`, or `DENSE_RANK()` combined with `PARTITION BY` to rank items within categories.

## Why Interviewers Ask This
* It is extremely common in real-world dashboards (e.g., "top 5 customers", "top 3 products per category").
* It effectively tests a candidate's mastery of Window Functions and Common Table Expressions (CTEs).
* It assesses your attention to detail regarding edge cases, specifically how you handle data ties.

## Core Concepts
* Window functions execute after the `WHERE` clause, so you cannot filter on a window function result directly; you must wrap it in a CTE or subquery first.
* The `OVER()` clause dictates the window: `PARTITION BY` groups the data, and `ORDER BY` determines the ranking logic.

## When to Use
* Any prompt containing keywords like "top N", "bottom N", "best", or "worst".
* When asked to return the highest-ranked items either globally or broken down by a specific dimension.

## Limitations & Nuances
* SQL dialects handle global Top N differently: PostgreSQL/MySQL use `LIMIT`, SQL Server uses `TOP N`, and Oracle uses `FETCH FIRST N ROWS ONLY`.
* `ROW_NUMBER()` will arbitrarily drop tied rows if the limit is reached, which might not reflect business reality.

## Common Comparisons
* **ROW_NUMBER():** Assigns strictly unique, sequential numbers (e.g., 1, 2, 3). Ties are broken arbitrarily.
* **RANK():** Gives ties the same rank, but *skips* the subsequent ranks (e.g., 1, 1, 3).
* **DENSE_RANK():** Gives ties the same rank and does *not* skip subsequent ranks (e.g., 1, 1, 2).

## Common Interview Traps
* Using `LIMIT` when the prompt asks for a Top N *per group*.
* Forgetting the `PARTITION BY` clause in the window function, resulting in a global rank instead of a group rank.
* Omitting the `ORDER BY` inside the `OVER()` clause, which causes a random, meaningless ranking.
* Trying to filter the rank directly in the `WHERE` clause instead of evaluating it in a CTE first.

## SQL Syntax (Top N Per Group)
```sql
WITH ranked_products AS (
    SELECT 
        product_id, 
        category, 
        revenue,
        DENSE_RANK() OVER(PARTITION BY category ORDER BY revenue DESC) AS rank_num
    FROM products
)
SELECT * 
FROM ranked_products 
WHERE rank_num <= 3;
```

## 45-Second Interview Answer
"To find Top N records, the approach depends on whether we need global or grouped results. For a global Top N, I simply use `ORDER BY` combined with `LIMIT`. However, for Top N per group, like top 3 products per category, I use a window function inside a CTE. I usually prefer `DENSE_RANK()` over `ROW_NUMBER()` to ensure tied records are treated fairly without skipping rank sequence. Once ranked in the CTE, I filter the result in the main query using a `WHERE` clause."

---


## Example Questions and Answers

**Q1. Find the top 5 customers by total purchase amount.**
* **Ideal Answer:**
  ```sql
  SELECT customer_id, SUM(purchase_amount) as total_amount
  FROM purchases
  GROUP BY customer_id
  ORDER BY total_amount DESC
  LIMIT 5;
  ```
* **Common Mistake:** Forgetting to aggregate (`SUM`) the purchase amounts before ordering.
* **Follow-up:** How would you write this if the database was SQL Server instead of Postgres? *(Answer: I would replace `LIMIT 5` at the end with `SELECT TOP 5 customer_id...` at the beginning of the query).*

**Q2. Find the top 3 sales representatives per region by number of deals closed.**
* **Ideal Answer:**
  ```sql
  WITH RepRanks AS (
      SELECT rep_id, region, deals_closed,
             DENSE_RANK() OVER(PARTITION BY region ORDER BY deals_closed DESC) as rep_rank
      FROM sales_reps
  )
  SELECT rep_id, region, deals_closed
  FROM RepRanks
  WHERE rep_rank <= 3;
  ```
* **Common Mistake:** Using `LIMIT 3` which only returns 3 rows total, not 3 per region.
* **Follow-up:** Why did you use `DENSE_RANK()` instead of `ROW_NUMBER()`? *(Answer: If two reps tie for 3rd place, DENSE_RANK() includes them both, which is usually the desired business logic for leaderboards).*

**Q3. Get the bottom 5 products by revenue.**
* **Ideal Answer:**
  ```sql
  SELECT product_id, product_name, revenue
  FROM products
  ORDER BY revenue ASC
  LIMIT 5;
  ```
* **Common Mistake:** Using `DESC` out of habit, which fetches the top 5 instead of the bottom 5.
* **Follow-up:** What if some products have NULL revenue? How do you ensure they don't fill up the bottom 5? *(Answer: I would add `WHERE revenue IS NOT NULL` before the ORDER BY clause).*

**Q4. Find the top 2 most ordered products in each product category.**
* **Ideal Answer:**
  ```sql
  WITH CategoryRanks AS (
      SELECT category, product_id, order_count,
             DENSE_RANK() OVER(PARTITION BY category ORDER BY order_count DESC) as rnk
      FROM products
  )
  SELECT category, product_id, order_count
  FROM CategoryRanks
  WHERE rnk <= 2;
  ```
* **Common Mistake:** Trying to filter `WHERE DENSE_RANK() OVER(...) <= 2` directly in the main query, causing a syntax error.
* **Follow-up:** What happens to the ranking if you forget `PARTITION BY`? *(Answer: The window function would rank the products globally across the entire table, rather than resetting the rank for each category).*

**Q5. Find employees with the 3 highest salaries in each department.**
* **Ideal Answer:**
  ```sql
  WITH SalaryRanks AS (
      SELECT emp_id, department, salary,
             DENSE_RANK() OVER(PARTITION BY department ORDER BY salary DESC) as rank_val
      FROM employees
  )
  SELECT emp_id, department, salary
  FROM SalaryRanks
  WHERE rank_val <= 3;
  ```
* **Common Mistake:** Using `RANK()`. If there is a tie for 1st place, `RANK()` assigns 1, 1, 3. The person with the actual 2nd highest salary gets assigned rank 3, which can cause them to be pushed out if filtering for top 2.
* **Follow-up:** Can we achieve this without a CTE? *(Answer: Yes, you can use a subquery in the FROM clause, but CTEs are generally preferred for readability).*

## Practice Questions:

In [1]:
import sqlite3
import pandas as pd

conn= sqlite3.connect('/home/shail/interview-prep/01_SQL/oracle_hr.db')

### Q1:

Using employees table from HR Schema:

"Write a SQL query to find the employees holding the top 2 highest distinct salary amounts in each department. If multiple employees share that same salary, include all of them. Return the department, emp_id, and salary."

In [3]:
pd.read_sql_query(sql= """
select * from employees limit 5;
""", con= conn)

,employee_id,first_name,last_name,email,phone_number,hire_date,job_id,salary,commission_pct,manager_id,department_id
0,100,Steven,King,SKING,1.515.555.0100,2013-06-17,AD_PRES,24000.0,None,NaN,90
1,101,Neena,Yang,NYANG,1.515.555.0101,2015-09-21,AD_VP,17000.0,None,100.0,90
2,102,Lex,Garcia,LGARCIA,1.515.555.0102,2011-01-13,AD_VP,17000.0,None,100.0,90
3,103,Alexander,James,AJAMES,1.590.555.0103,2016-01-03,IT_PROG,9000.0,None,102.0,60
4,104,Bruce,Miller,BMILLER,1.590.555.0104,2017-05-21,IT_PROG,6000.0,None,103.0,60


In [6]:
pd.read_sql_query(sql= """
with sal_rank as (
    select department_id, employee_id, salary,
    dense_rank() over(partition by department_id order by salary desc) as rnk
    from employees
)
select department_id, employee_id, salary
from sal_rank
where rnk < 3
""", con= conn)

,department_id,employee_id,salary
0,NaN,178,7000.0
1,10.0,200,4400.0
2,20.0,201,13000.0
3,20.0,202,6000.0
4,30.0,114,11000.0
5,30.0,115,3100.0
6,40.0,203,6500.0
7,50.0,121,8200.0
8,50.0,120,8000.0
9,60.0,103,9000.0
